In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from itertools import product
from tqdm import tqdm 
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline      import Pipeline

In [2]:
def load_trim(path: Path | str) -> pd.DataFrame:
    return (
        pd.read_csv(path)
          .iloc[8:]                 # skip rows 0‥7
          .reset_index(drop=True)   # tidy, so index starts at 0
    )

In [3]:
DATA_DIR = Path(".")
train_df = pd.read_csv(DATA_DIR / "feature_engineered_final_train_data.csv")
val_df   = pd.read_csv(DATA_DIR / "feature_engineered_final_val_data.csv")
test_df  = pd.read_csv(DATA_DIR / "feature_engineered_final_test_data.csv")

TARGET = "Mood"

/var/folders/lc/9q2gf4dn4snfnltmj87764500000gn/T/ipykernel_91988/3145008697.py:2: DtypeWarning: Columns (58) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv(DATA_DIR / "feature_engineered_final_train_data.csv")
/var/folders/lc/9q2gf4dn4snfnltmj87764500000gn/T/ipykernel_91988/3145008697.py:3: DtypeWarning: Columns (52) have mixed types. Specify dtype option on import or set low_memory=False.
  val_df   = pd.read_csv(DATA_DIR / "feature_engineered_final_val_data.csv")
/var/folders/lc/9q2gf4dn4snfnltmj87764500000gn/T/ipykernel_91988/3145008697.py:4: DtypeWarning: Columns (52) have mixed types. Specify dtype option on import or set low_memory=False.
  test_df  = pd.read_csv(DATA_DIR / "feature_engineered_final_test_data.csv")


In [4]:
for d in (train_df, val_df, test_df):
    d.drop(columns=["timestamp"], inplace=True, errors="ignore")

In [5]:
train_cols = list(train_df.columns)                 
num_medians = train_df.median(numeric_only=True)    
cat_modes   = {
    col: train_df[col].mode(dropna=True)[0]
    for col in train_df.select_dtypes(include="object").columns
}

for name, df in (("val", val_df), ("test", test_df)):
    missing = set(train_cols) - set(df.columns)
    if missing:
        for col in missing:
            if col in num_medians:                 # numeric → median
                df[col] = num_medians[col]
            else:                                  # categorical → mode
                df[col] = cat_modes.get(col, "")
        df = df[train_cols]

    if name == "val":
        val_df = df
    else:
        test_df = df

In [6]:
def split_xy(df: pd.DataFrame):
    X = df.drop(columns=[TARGET]).copy()
    y = df[TARGET].astype(int).to_numpy()
    return X, y

In [7]:
cat_cols  = ["Activity"]                           
for d in (train_df, val_df, test_df):
    for col in d.columns:
        if "timestamp" in col.lower():
            d.drop(columns=col, inplace=True)

    if "Common time (s)" in d.columns:
        d["Common time (s)"] = pd.to_numeric(
            d["Common time (s)"], errors="coerce"
        )
    stray_obj = [
        c for c in d.select_dtypes(include="object").columns
        if c not in cat_cols
    ]
    if stray_obj:
        print("⚠️  Dropping non-numeric, non-categorical cols:", stray_obj)
        d.drop(columns=stray_obj, inplace=True)

num_cols = [
    c for c in train_df.columns
    if c not in cat_cols + [TARGET]        ]

⚠️  Dropping non-numeric, non-categorical cols: ['mag_saturation']
⚠️  Dropping non-numeric, non-categorical cols: ['mag_saturation']
⚠️  Dropping non-numeric, non-categorical cols: ['mag_saturation']


In [8]:
num_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),   # ← NEW
    ("scale",  StandardScaler())
])

cat_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),  # ← NEW
    ("oh",     OneHotEncoder(handle_unknown="ignore"))
])

preproc = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
], remainder="drop")

In [9]:
class EchoStateNetwork:
    def __init__(self, 
                 n_in, 
                 n_res=500, 
                 sr=0.9, 
                 sparsity=0.01,
                 leak=1.0, 
                 random_state=42,
                 clf=LogisticRegression(max_iter=500,
                                        multi_class="multinomial")):
        self.n_in, self.n_res = n_in, n_res
        self.sr, self.sparsity, self.leak = sr, sparsity, leak
        self.rng = np.random.default_rng(random_state)

        self.W_in = self.rng.uniform(-1, 1, (n_res, n_in + 1))
        W = self.rng.uniform(-1, 1, (n_res, n_res))
        W[self.rng.random(W.shape) > sparsity] = 0.0
        eig_max = np.max(np.abs(np.linalg.eigvals(W)))
        self.W = W * (sr / eig_max)

        self.clf = clf

    def _run(self, X, r0=None):
        T = X.shape[0]
        R = np.zeros((T, self.n_res))
        r = np.zeros(self.n_res) if r0 is None else r0
        for t in range(T):
            u = np.hstack((1.0, X[t]))
            r_tilde = np.tanh(self.W_in @ u + self.W @ r)
            r = (1 - self.leak) * r + self.leak * r_tilde
            R[t] = r
        return R

    def fit(self, X, y, washout=50):
        R = self._run(X)[washout:]
        self.clf.fit(R, y[washout:])
        return self

    def predict(self, X):
        R = self._run(X)
        return self.clf.predict(R)

In [10]:
X_train_df, y_train = split_xy(train_df)
X_val_df,   y_val   = split_xy(val_df)
X_test_df,  y_test  = split_xy(test_df)

preproc.fit(X_train_df)                 
X_train = preproc.transform(X_train_df)
X_val   = preproc.transform(X_val_df)
X_test  = preproc.transform(X_test_df)

/Users/aylinogras/Desktop/MLQS/ML_for_Sensor_data-1/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:637: UserWarning: Skipping features without any observed values: ['Common time (s)']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/aylinogras/Desktop/MLQS/ML_for_Sensor_data-1/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:637: UserWarning: Skipping features without any observed values: ['Common time (s)']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/aylinogras/Desktop/MLQS/ML_for_Sensor_data-1/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:637: UserWarning: Skipping features without any observed values: ['Common time (s)']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Users/aylinogras/Desktop/MLQS/ML_for_Sensor_data-1/.venv/lib/python3.12/site-packages/sklearn/impute/_base.py:637: Us

In [11]:
params = dict(
    n_res     = 600,   
    sr        = 0.9,   
    leak      = 0.5,   
    ridge     = 1e-4,  
    washout   = 50     
)

single_esn = EchoStateNetwork(
    n_in  = X_train.shape[1],
    n_res = params["n_res"],
    sr    = params["sr"],
    leak  = params["leak"],
    random_state = 42,
    clf   = LogisticRegression(
               max_iter = 500,
               C        = 1 / params["ridge"],
               multi_class = "multinomial"
           )
)

single_esn.fit(X_train, y_train, washout=params["washout"])

val_pred = single_esn.predict(X_val)
val_acc  = accuracy_score(y_val, val_pred)
val_f1   = f1_score(y_val, val_pred, average="macro")

print(
    f"Fixed-ESN  |  n_res={params['n_res']}, sr={params['sr']}, "
    f"leak={params['leak']}, ridge={params['ridge']}, "
    f"washout={params['washout']}\n"
    f"Validation accuracy={val_acc:.3f},  macro-F1={val_f1:.3f}"
)

"""
Fixed-ESN  |  n_res=600, sr=0.9, leak=0.5, ridge=0.0001, washout=50
Validation accuracy=0.502,  macro-F1=0.223
"""

/Users/aylinogras/Desktop/MLQS/ML_for_Sensor_data-1/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.
  warnings.warn(


Fixed-ESN  |  n_res=600, sr=0.9, leak=0.5, ridge=0.0001, washout=50
Validation accuracy=0.502,  macro-F1=0.223


'\nFixed-ESN  |  n_res=600, sr=0.9, leak=0.5, ridge=0.0001, washout=50\nValidation accuracy=0.502,  macro-F1=0.223\n'

### hp tuning

In [120]:
grid = {
    # reservoir size
    #"n_res":   [300, 600, 900],          # small · medium · large
    # spectral radius
    #"sr":      [0.8, 1.0, 1.2],          # centre + one on each side
    # leak rate
    "leak":    [0.3, 0.6, 0.9],          # slow · mid · fast
    # ℓ2 on read-out  (C = 1/ridge inside the loop)
    "ridge":   [1e-5, 1e-3, 1e-1],       # weak · medium · strong
    # wash-out steps
    "washout": [50, 100]                 
}

search_space = list(product(
    #grid["n_res"],
    #grid["sr"],
    grid["leak"],
    grid["ridge"],
    grid["washout"]
))

best_esn, best_acc = None, -1

n_res = 600 
sr = 1.0

#for n_res, sr, leak, ridge, washout in tqdm(search_space, desc="Grid-search"):
for leak, ridge, washout in tqdm(search_space, desc="Grid-search"):
    esn = EchoStateNetwork(
        n_in  = X_train.shape[1],
        n_res = n_res,
        sr    = sr,
        leak  = leak,
        random_state = 42,
        clf = LogisticRegression(
            max_iter=500,
            C=1 / ridge,                 # inverse λ
            multi_class="multinomial"
        )
    )
    esn.fit(X_train, y_train, washout=washout)
    acc = accuracy_score(y_val, esn.predict(X_val))

    if acc > best_acc:
        best_esn, best_acc = esn, acc
        best_cfg = (n_res, sr, leak, ridge, washout)

print(
    f"\nBest-val acc={best_acc:.3f} with "
    f"n_res={best_cfg[0]}, sr={best_cfg[1]}, "
    f"leak={best_cfg[2]}, ridge={best_cfg[3]}, washout={best_cfg[4]}"
)

# # Best-val acc=0.502 with n_res=600, sr=1.0, leak=0.9, ridge=1e-05, washout=50


Grid-search:   0%|          | 0/18 [00:00<?, ?it/s]/Users/aylinogras/Desktop/MLQS/ML_for_Sensor_data-1/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.
  warnings.warn(
Grid-search:   6%|▌         | 1/18 [01:04<18:12, 64.25s/it]/Users/aylinogras/Desktop/MLQS/ML_for_Sensor_data-1/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1254: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, binary problems will be fit as proper binary  logistic regression models (as if multi_class='ovr' were set). Leave it to its default value to avoid this warning.
  warnings.warn(
Grid-search:  11%|█         | 2/18 [03:26<29:24, 110.25s/it]/Users/aylinogra


Best-val acc=0.502 with n_res=600, sr=1.0, leak=0.9, ridge=1e-05, washout=50


In [ ]:
best_cfg = (600, 1.0, 0.9, 1e-5, 50)   # (n_res, sr, leak, ridge, washout)
best_n_res, best_sr, best_leak, best_ridge, best_washout = best_cfg
X_comb = np.vstack([X_train, X_val])
y_comb = np.hstack([y_train, y_val])

final_esn = EchoStateNetwork(
    n_in   = X_comb.shape[1],
    n_res  = best_n_res,
    sr     = best_sr,
    leak   = best_leak,
    random_state = 42,
    clf    = LogisticRegression(
                max_iter=500,
                C=1 / best_ridge,
                multi_class="multinomial"))

final_esn.fit(X_comb, y_comb, washout=best_washout)

test_pred = final_esn.predict(X_test)
print(
    "TEST  accuracy:",
    accuracy_score(y_test, test_pred),
    "macro-F1:",
    f1_score(y_test, test_pred, average="macro"))